# Create a custom PFLOTRAN reaction sandbox

This notebook helps you:
1. Check whether reaction species already exist in `hanford.dat`
2. Add any missing **basis** species (a0, charge, molecular weight)
3. Define stoichiometry and rate / water-activity parameters
4. Generate `sandbox/custom_<timestamp>/` with a tailored `hanford.dat`, `.F90` module, and `.in` snippet

**Kinetic form (v1):** power-law over reactants × optional water-activity inhibition (same pattern as AWINHIBIT).

Edit the Python cells below, then run top-to-bottom.

## 1. Load `hanford.dat`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Allow running from repo root or from sandbox/
SANDBOX = Path.cwd()
if (SANDBOX / "custom_sandbox_gen.py").exists():
    pass
elif (SANDBOX / "sandbox" / "custom_sandbox_gen.py").exists():
    SANDBOX = SANDBOX / "sandbox"
else:
    raise FileNotFoundError("Run this notebook from the repo root or sandbox/")

sys.path.insert(0, str(SANDBOX))
from custom_sandbox_gen import (
    NewSpecies,
    ReactionDef,
    parse_hanford_dat,
    reaction_equation,
    render_f90,
    render_in_snippet,
    search_species,
    species_status,
    write_custom_sandbox,
)

HANFORD = SANDBOX / "hanford.dat"
parsed = parse_hanford_dat(HANFORD)
print(f"Loaded {HANFORD}")
print(f"  basis: {len(parsed.basis)}")
print(f"  aqueous complexes: {len(parsed.aqueous)}")
print(f"  gases: {len(parsed.gases)}")
print(f"  minerals: {len(parsed.minerals)}")

### Optional: search the database
Change `QUERY` to explore existing names (exact spelling matters later).

In [ ]:
QUERY = "CH4"  # try: Acetate, H2, SO4, DOM, ...
hits = search_species(parsed, QUERY, limit=40)
print(f"{len(hits)} hit(s) for {QUERY!r}:")
for name in hits:
    print(" ", name)

## 2. Name the sandbox

Use a short lowercase name (`[a-z][a-z0-9_]*`). It becomes:
- keyword in `.in`: `MYREACT` (underscores stripped, uppercased)
- file: `reaction_sandbox_my_react.F90`
- module: `Reaction_Sandbox_MyReact_class`

In [ ]:
SANDBOX_NAME = "demo_h2"  # change me
DESCRIPTION = "Example hydrogenotrophic methanogenesis with a_w inhibition"

## 3. Define stoichiometry

Dictionaries map **species name → stoichiometric coefficient** (must be > 0).
`H2O` is allowed for documentation but is treated as solvent (no residual update).

Rate law uses **reactants only**: `k * Π(C_i^ν_i) * f(a_w) * L_water`.

In [ ]:
# Example: 4 H2(aq) + HCO3- + H+ -> CH4(aq) + 3 H2O
REACTANTS = {
    "H2(aq)": 4,
    "HCO3-": 1,
    "H+": 1,
}
PRODUCTS = {
    "CH4(aq)": 1,
    "H2O": 3,
}

print(reaction_equation(REACTANTS, PRODUCTS))

## 4. Species audit

| Status | Meaning |
|--------|---------|
| `in_basis` | Ready to use as a primary species |
| `in_aqueous_complexes` / `in_gases` / … | Name exists in the DB; may still need to appear under `PRIMARY_SPECIES` in your deck |
| `solvent` | `H2O` — skipped in Fortran residuals |
| `missing` | Must be added below as a new basis species |

In [ ]:
rxn_names = set(REACTANTS) | set(PRODUCTS)
audit_df = pd.DataFrame(species_status(rxn_names, parsed))
display(audit_df)

missing = audit_df.loc[audit_df["status"] == "missing", "species"].tolist()
if missing:
    print("Missing from hanford.dat — add them in the next cell:", missing)
else:
    print("All reaction species are present in hanford.dat (or are solvent).")

## 5. Add missing basis species (if any)

Each entry becomes a basis line: `'Name' a0 charge molecular_weight`.
Leave the list empty if nothing is missing.

In [ ]:
NEW_SPECIES = [
    # Example (uncomment / edit if needed):
    # NewSpecies(name="MyNewSpecies", a0=3.0, charge=0.0, molecular_weight=100.0),
]

# Quick check: every missing name must appear in NEW_SPECIES
provided = {sp.name for sp in NEW_SPECIES}
still = [m for m in missing if m not in provided]
if still:
    print("Still need NewSpecies entries for:", still)
else:
    print(f"OK — {len(NEW_SPECIES)} new species will be appended to hanford.dat")

## 6. Rate / inhibition defaults

These values go into `reaction_def.json` and the `.in` snippet.
The Fortran module still **reads** them at runtime from the input deck.

In [ ]:
RATE_CONSTANT = 1.0e-10          # mol/m^3-sec
AW_THRESHOLD = 0.5               # water activity cutoff
INHIBITION_TYPE = "THRESHOLD"    # or "SMOOTHSTEP"

# Optional Arrhenius (set both, or leave as None)
ACTIVATION_ENERGY = None         # J/mol
REFERENCE_TEMPERATURE = None     # C

defn = ReactionDef(
    name=SANDBOX_NAME,
    description=DESCRIPTION,
    reactants=REACTANTS,
    products=PRODUCTS,
    new_species=NEW_SPECIES,
    rate_constant=RATE_CONSTANT,
    aw_threshold=AW_THRESHOLD,
    inhibition_type=INHIBITION_TYPE,
    activation_energy=ACTIVATION_ENERGY,
    reference_temperature=REFERENCE_TEMPERATURE,
)
defn.validate()
print("ReactionDef OK:", reaction_equation(defn.reactants, defn.products))

## 7. Preview generated `.in` snippet and Fortran residuals

In [ ]:
print("=== reaction_sandbox_block.in ===")
print(render_in_snippet(defn))

f90 = render_f90(defn)
# Show the rate + residual section for a quick visual check
start = f90.find("reaction_rate = rate_constant")
end = f90.find("if (compute_derivative)")
print("=== Evaluate rate / residuals (excerpt) ===")
print(f90[start:end].rstrip())

## 8. Generate `sandbox/custom_<timestamp>/`

Writes `hanford.dat`, `reaction_sandbox_*.F90`, `reaction_sandbox_block.in`, `reaction_def.json`, and `README.md`.

In [ ]:
out_dir = write_custom_sandbox(defn, out_root=SANDBOX, hanford_path=HANFORD)
print(f"Wrote: {out_dir}")
for p in sorted(out_dir.iterdir()):
    print(f"  {p.name}")
print()
print("Next steps:")
print(f"  1. Point DATABASE at {out_dir / 'hanford.dat'}")
print(f"  2. Paste {out_dir / 'reaction_sandbox_block.in'} into your .in chemistry section")
print("  3. Patch + rebuild PFLOTRAN:")
print("     python3 scripts/patch_pflotran_sandboxes.py \\")
print("       --pflotran-src /path/to/pflotran/src/pflotran \\")
print("       --sandbox-dir sandbox/ \\")
print(f"       --extra-sandbox-dir {out_dir.relative_to(SANDBOX.parent) if (SANDBOX.parent / 'scripts').exists() else out_dir}")